In [ ]:
import pandas as pd
print(pd.__version__)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# load data
df = pd.read_json("anilist_5000_with_image.json")

# fix title 
df["title"] = df["title"].apply(
    lambda t: t["english"] if t["english"] else t["romaji"]
)

# clean description
df["description"] = df["description"].fillna("")
df["description"] = df["description"].apply(
    lambda x: re.sub(r"<.*?>", "", x)
)

# clean titles
df["title"] = df["title"].fillna("").astype(str)

# clean genres
df["genres_text"] = df["genres"].apply(
    lambda g: " ".join(g) if isinstance(g, list) else ""
)


# features
df["features"] = (df["title"] + " " + df["genres_text"] + " " + df["genres_text"] + " " + df["description"])
# read first rows of how combined features look
print(df[["title", "genres_text", "features"]].head())


# vectorizer 
# turn text into number
vectorizer = TfidfVectorizer()

# create a matrix 
# rows = animes 
# columns = words
# values = TF-IDF scores
tfidf_matrix = vectorizer.fit_transform(df["features"])
print(tfidf_matrix.shape)

# compare anime to anime with cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(similarity_matrix.shape)
# compare first anime with the rest of animes
print(similarity_matrix[0])
# title of first anime
print(df.iloc[0]["title"])


def recommend(title, top_n = 5):

    # find the index
    matches = df[df["title"].str.lower() == title.lower()]

    if matches.empty:
        return f"Title '{title}' not found."
    
    anime_index = matches.index[0]

    # get similarity scores
    # pair index + score
    similarity_scores = list(enumerate(similarity_matrix[anime_index]))
    # sort scores
    sorted_similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    # remove the current title and take the next 5
    top_results = sorted_similarity_scores[1:top_n+1]
    # return those titles
    recommended_titles = [(df.iloc[i[0]]["title"], i[1]) for i in top_results]

    return recommended_titles

# WITH DATASET MOVIEMUSE ONLY

# print columns
# Index(['id', 'title', 'episodes', 'imageURL', 'externalId', 'source', 'genres',
#        'status', 'description', 'inWatchlist', 'type', 'reviews'],
#       dtype='str')
# print(df.columns)
# #filter anime
# anime_df = df[df["type"] == "ANIME"].copy()
# # make sure to put empty string when field is empty
# # clean description data more
# anime_df["description"] = anime_df["description"].fillna("")
# anime_df["description"] = anime_df["description"].apply(
#     lambda text: re.sub(r"<.*?>", " ", text) if isinstance(text, str) else ""
# )
# anime_df["description"] = anime_df["description"].apply(
#     lambda text: re.sub(r"\(Source:.*?\)", "", text) if isinstance(text, str) else ""
# )
# # make sure title is a string
# anime_df["title"] = anime_df["title"].fillna("").astype(str)
# # turn genres into a strings
# anime_df["genres_text"] = anime_df["genres"].apply(lambda g: " ".join(g) if isinstance(g, list) else "")


# first_rows = anime_df.head()
# print(first_rows)

# # features
# anime_df["features"] = (anime_df["title"] + " " + anime_df["genres_text"] + " " + anime_df["genres_text"] + " " + anime_df["description"])
# # read first rows of how combined features look
# print(anime_df[["title", "genres_text", "features"]].head())
# print(anime_df[["features"]].head())

# # vectorizer 
# # turn text into number
# vectorizer = TfidfVectorizer()

# # create a matrix 
# # rows = animes 
# # columns = words
# # values = TF-IDF scores
# tfidf_matrix = vectorizer.fit_transform(anime_df["features"])
# print(tfidf_matrix.shape)

# # compare anime to anime with cosine similarity
# similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
# print(similarity_matrix.shape)
# # compare first anime with the rest of animes
# print(similarity_matrix[0])
# # title of first anime
# print(anime_df.iloc[0]["title"])


# def recommend(title, top_n = 5):

#     # find the index
#     # anime_index = anime_df[anime_df["title"] == "Naruto"].index
#     anime_index = anime_df[anime_df["title"] == title].index[0]
#     # get similarity scores
#     # similarity_scores = similarity_matrix[4]
#     similarity_scores = similarity_matrix[anime_index]
#     # pair index + score
#     similarity_scores = list(enumerate(similarity_scores))
#     # sort scores
#     sorted_similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
#     # remove the current title and take the next 5
#     top_results = sorted_similarity_scores[1:top_n+1]
#     # return those titles
#     recommended_titles = [(anime_df.iloc[i[0]]["title"], i[1]) for i in top_results]

#     return recommended_titles

print(recommend("Naruto"))
print(recommend("Tokyo Revengers"))
print(recommend("A Silent Voice"))

3.0.1
                            title  \
0                 Attack on Titan   
1  Demon Slayer: Kimetsu no Yaiba   
2                      Death Note   
3                  JUJUTSU KAISEN   
4                My Hero Academia   

                                   genres_text  \
0                 Action Drama Fantasy Mystery   
1  Action Adventure Drama Fantasy Supernatural   
2  Mystery Psychological Supernatural Thriller   
3                    Action Drama Supernatural   
4                      Action Adventure Comedy   

                                            features  
0  Attack on Titan Action Drama Fantasy Mystery A...  
1  Demon Slayer: Kimetsu no Yaiba Action Adventur...  
2  Death Note Mystery Psychological Supernatural ...  
3  JUJUTSU KAISEN Action Drama Supernatural Actio...  
4  My Hero Academia Action Adventure Comedy Actio...  
(5000, 28324)
(5000, 5000)
[1.         0.07191638 0.05418764 ... 0.05232198 0.04571016 0.02209507]
Attack on Titan
[('Naruto: Shippuden', np